
![example](images/aircraft.jpg)


# Potential Risks of aircrafts

**Authors:** Jose Ortega
***

## Overview

The task of this project is to analyse the data (aviation-accident-Database) in order to generate recommendatios for a business stakeholder.

## Business Problem

The company is looking in to new industries to diversify its portfolio, they want to purchase and operate airplanes for commercial and private enterprises, they do not know much about the potential risk of aircrafts. My task is to determine which aircrafts are the lowest risk for the company to start this new business endeavor.

## Data Understanding

The data used in this project is from the NTSB (National Transportation and Safety Board), that contains information from 1962 and later about civil aviation accidents and selected incidents within tha United States, its territories and possessions, and international waters.
Some of the important variables in the dataset are : Aircraft damage, category, make, model, number of engines, engine type, injury severity and total injuries.

In [241]:
# Import standard packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


In [242]:
df = pd.read_csv('./Data/AviationData.csv', encoding='latin1') # loading 'AviationData' as DataFrame

df.head() # to see how the data looks like

/var/folders/9f/w2s6x31n56v6ghk0qyt8tvx80000gn/T/ipykernel_35857/965402264.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./Data/AviationData.csv', encoding='latin1') # loading 'AviationData' as DataFrame


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [243]:
df.shape # to see how many columns and rows are in the DataFrame

(88889, 31)

In [244]:
df.info() # to see the information about the DataFrame

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

We can see now that we are working with a large DataFrame , most of the columns have missing values , this is something that we need to consider.
We can also see that the columns names have dots between the words , we can replace the dots for underscore for easy reading.

In [246]:
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [247]:
df.columns = df.columns.str.replace('.','_') # replacing the dots in the columns for underscores

Now lets have a look of the data types in the DataFrame.

In [249]:
df.dtypes

Event_Id                   object
Investigation_Type         object
Accident_Number            object
Event_Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport_Code               object
Airport_Name               object
Injury_Severity            object
Aircraft_damage            object
Aircraft_Category          object
Registration_Number        object
Make                       object
Model                      object
Amateur_Built              object
Number_of_Engines         float64
Engine_Type                object
FAR_Description            object
Schedule                   object
Purpose_of_flight          object
Air_carrier                object
Total_Fatal_Injuries      float64
Total_Serious_Injuries    float64
Total_Minor_Injuries      float64
Total_Uninjured           float64
Weather_Condition          object
Broad_phase_of_flight      object
Report_Status 

## Data Preparation

We can see that most of the columns have the right datatype (object,float) , the Event_Date has an object datatype instead of a datetime datatype, lets change this.

In [252]:
df['Event_Date'] = pd.to_datetime(df['Event_Date']) #converting the datatype to datetime

In [253]:
df['Event_Date'].head() # dtype now is datetime

0   1948-10-24
1   1962-07-19
2   1974-08-30
3   1977-06-19
4   1979-08-02
Name: Event_Date, dtype: datetime64[ns]

Now I would like to work with the year of the date (I dont really need month and day for the analysis).
To do that I will create another column called 'Event_Year' , in there we will display just the year. 

In [255]:
df['Event_Year'] = df['Event_Date'].dt.year # creating the new column ['Eventy_Year'] with only year values
df['Event_Year'].head() # checking the result 

0    1948
1    1962
2    1974
3    1977
4    1979
Name: Event_Year, dtype: int32

Now it is time to drop the colummns we are not going to use for the analysis, there are quite a few columns that have more than 50% of missing values like Latitude, Longitud , Aircraft_Category, Far_Description , etc 
The columns that I going to use for analysis are :
Investigation_Type, Injury_Severity, Aircraft_damage, Aircraft_Category, Make, Model, Amateur_Built, Number_of_Engines, Engine_Type, Purpose_of_flight, Total_Fatal_Injuries, Total_Serious_Injuries, Total_Minor_Injuries, Total_Uninjured, Weather_Condition, Broad_phase_of_flight and the recently column I just created Event Year.

In [257]:
# creating a list the columns we are going to use 

columns_to_use = ['Investigation_Type', 'Injury_Severity', 'Aircraft_damage', 'Aircraft_Category', 'Make', 'Model'
                  , 'Amateur_Built', 'Number_of_Engines', 'Engine_Type', 'Purpose_of_flight', 'Total_Fatal_Injuries'
                  , 'Total_Serious_Injuries', 'Total_Minor_Injuries', 'Total_Uninjured', 'Weather_Condition', 'Broad_phase_of_flight',
                  'Event_Year']

df_relevant_columns = df[columns_to_use]
df_relevant_columns
                  

,Investigation_Type,Injury_Severity,Aircraft_damage,Aircraft_Category,Make,Model,Amateur_Built,Number_of_Engines,Engine_Type,Purpose_of_flight,Total_Fatal_Injuries,Total_Serious_Injuries,Total_Minor_Injuries,Total_Uninjured,Weather_Condition,Broad_phase_of_flight,Event_Year
0,Accident,Fatal(2),Destroyed,NaN,Stinson,108-3,No,1.0,Reciprocating,Personal,2.0,0.0,0.0,0.0,UNK,Cruise,1948
1,Accident,Fatal(4),Destroyed,NaN,Piper,PA24-180,No,1.0,Reciprocating,Personal,4.0,0.0,0.0,0.0,UNK,Unknown,1962
2,Accident,Fatal(3),Destroyed,NaN,Cessna,172M,No,1.0,Reciprocating,Personal,3.0,NaN,NaN,NaN,IMC,Cruise,1974
3,Accident,Fatal(2),Destroyed,NaN,Rockwell,112,No,1.0,Reciprocating,Personal,2.0,0.0,0.0,0.0,IMC,Cruise,1977
4,Accident,Fatal(1),Destroyed,NaN,Cessna,501,No,NaN,NaN,Personal,1.0,2.0,NaN,0.0,VMC,Approach,1979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88884,Accident,Minor,NaN,NaN,PIPER,PA-28-151,No,NaN,NaN,Personal,0.0,1.0,0.0,0.0,NaN,NaN,2022
88885,Accident,NaN,NaN,NaN,BELLANCA,7ECA,No,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,2022
88886,Accident,Non-Fatal,Substantial,Airplane,AMERICAN CHAMPION AIRCRAFT,8GCBC,No,1.0,NaN,Personal,0.0,0.0,0.0,1.0,VMC,NaN,2022
88887,Accident,NaN,NaN,NaN,CESSNA,210N,No,NaN,NaN,Personal,0.0,0.0,0.0,0.0,NaN,NaN,2022


In [258]:
df_relevant_columns.info() # now we have only 17 columns 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation_Type      88889 non-null  object 
 1   Injury_Severity         87889 non-null  object 
 2   Aircraft_damage         85695 non-null  object 
 3   Aircraft_Category       32287 non-null  object 
 4   Make                    88826 non-null  object 
 5   Model                   88797 non-null  object 
 6   Amateur_Built           88787 non-null  object 
 7   Number_of_Engines       82805 non-null  float64
 8   Engine_Type             81793 non-null  object 
 9   Purpose_of_flight       82697 non-null  object 
 10  Total_Fatal_Injuries    77488 non-null  float64
 11  Total_Serious_Injuries  76379 non-null  float64
 12  Total_Minor_Injuries    76956 non-null  float64
 13  Total_Uninjured         82977 non-null  float64
 14  Weather_Condition       84397 non-null

Let's have a look at the value count of "Aircraft_Category"

In [260]:
df_relevant_columns['Aircraft_Category'].value_counts()

Aircraft_Category
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

We are only interested on Airplane "category" , so we need to keep only the rows with airplane as category

In [262]:
df_relevant_columns = df_relevant_columns[df_relevant_columns['Aircraft_Category'] == 'Airplane']
df_relevant_columns.shape # now we can see we only have 27617 rows to work with

(27617, 17)

In [263]:
# Let's have a look at the info 
df_relevant_columns.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27617 entries, 5 to 88886
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation_Type      27617 non-null  object 
 1   Injury_Severity         26803 non-null  object 
 2   Aircraft_damage         26335 non-null  object 
 3   Aircraft_Category       27617 non-null  object 
 4   Make                    27608 non-null  object 
 5   Model                   27586 non-null  object 
 6   Amateur_Built           27600 non-null  object 
 7   Number_of_Engines       24863 non-null  float64
 8   Engine_Type             23391 non-null  object 
 9   Purpose_of_flight       23878 non-null  object 
 10  Total_Fatal_Injuries    24452 non-null  float64
 11  Total_Serious_Injuries  24393 non-null  float64
 12  Total_Minor_Injuries    24739 non-null  float64
 13  Total_Uninjured         26717 non-null  float64
 14  Weather_Condition       24564 non-null  obj

In [264]:
# Now let's have a look of the values in 'Amateur_Built'
df_relevant_columns['Amateur_Built'].value_counts()

Amateur_Built
No     24417
Yes     3183
Name: count, dtype: int64

Amateur Built are those planes constructed by individuals or organizations who are not engaged in professional aircraft construction.
For this reason we will drop those 3183 rows.

In [282]:
df_relevant_columns = df_relevant_columns[df_relevant_columns['Amateur_Built'] == 'No']
df_relevant_columns.shape

(24417, 17)

In [284]:
# Let's look at info now
df_relevant_columns.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24417 entries, 5 to 88886
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation_Type      24417 non-null  object 
 1   Injury_Severity         23604 non-null  object 
 2   Aircraft_damage         23147 non-null  object 
 3   Aircraft_Category       24417 non-null  object 
 4   Make                    24414 non-null  object 
 5   Model                   24399 non-null  object 
 6   Amateur_Built           24417 non-null  object 
 7   Number_of_Engines       21866 non-null  float64
 8   Engine_Type             20461 non-null  object 
 9   Purpose_of_flight       20734 non-null  object 
 10  Total_Fatal_Injuries    21651 non-null  float64
 11  Total_Serious_Injuries  21571 non-null  float64
 12  Total_Minor_Injuries    21855 non-null  float64
 13  Total_Uninjured         23699 non-null  float64
 14  Weather_Condition       21438 non-null  obj

In [286]:
# next step amateur_built , and lets keep cleaning the data
# df_relevant_columns['Investigation_Type'].value_counts()
df_relevant_columns['Investigation_Type'].value_counts()
# (df_relevant_columns.isnull().mean() * 100).sort_values(ascending=False)

Investigation_Type
Accident    22794
Incident     1623
Name: count, dtype: int64

## Data Modeling
Describe and justify the process for analyzing or modeling the data.

***
Questions to consider:
* How did you analyze or model the data?
* How did you iterate on your initial approach to make it better?
* Why are these choices appropriate given the data and the business problem?
***

In [ ]:
# Here you run your code to model the data


## Evaluation
Evaluate how well your work solves the stated business problem.

***
Questions to consider:
* How do you interpret the results?
* How well does your model fit your data? How much better is this than your baseline model?
* How confident are you that your results would generalize beyond the data you have?
* How confident are you that this model would benefit the business if put into use?
***

## Conclusions
Provide your conclusions about the work you've done, including any limitations or next steps.

***
Questions to consider:
* What would you recommend the business do as a result of this work?
* What are some reasons why your analysis might not fully solve the business problem?
* What else could you do in the future to improve this project?
***